# UVG-VCM → CCTV 규격 비트레이트 통제 데이터셋 구축
- 원본(4K, 60fps, 시퀀스당 약 30GB)은 저장하지 않고 **받으면서 바로** 720p·30fps 무손실 기준 영상으로 변환
- 기준 영상 → H.264 **4 / 2 / 1 / 0.5 Mbps** 각각 한 번씩 압축 (2-pass, GOP 250)
- 라벨: 원본 JSON 저장 + 720p·30fps COCO 변환
- 설정: Internet **On**, Accelerator **None**, **Save Version → Save & Run All**
- 중간에 끊겨도 다시 돌리면 끝난 시퀀스는 건너뜀 (이전 버전 Output을 Input으로 붙이고 `RESUME_FROM` 지정)

In [ ]:
%%writefile build_lib.py
"""UVG-VCM -> CCTV 규격 비트레이트 통제 데이터셋 구축 (Kaggle용)

시퀀스별 출력 ({OUT}/{seq}/):
  ref_720p30.mkv          무손실(FFV1) 기준 영상: 4K60 원본 -> 짝수 프레임만(30fps) -> 1280x720, yuv420p 8bit
  h264_{4000,2000,1000,500}k.mp4   기준 영상에서 각각 한 번씩 압축 (libx264, 2-pass, GOP 250)
  ann_raw.json            UVG-VCM 원본 라벨 그대로
  ann_720p30_coco.json    720p·30fps 기준 COCO 라벨 (변환 가능할 때)
  first.png / last.png    첫·마지막 프레임 미리보기 (고정 시점 확인용)
전체: {OUT}/meta.csv  (시퀀스별 비트 깊이, 단계별 실측 비트레이트·bpp, 라벨 통계)
"""
import csv, json, os, re, subprocess, collections
import numpy as np

W_SRC, H_SRC, FPS_SRC = 3840, 2160, 60
W_OUT, H_OUT, FPS_OUT = 1280, 720, 30
BITRATES_K = [4000, 2000, 1000, 500]
GOP = 250
SCALE = W_SRC / W_OUT  # 3.0


def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"FAILED: {cmd}\n{r.stderr[-3000:]}")
    return r.stdout


def detect_pixfmt(first_frame_u16):
    mx = int(first_frame_u16.max())
    if mx <= 1023:
        return "yuv444p10le", 10, mx
    if mx <= 4095:
        return "yuv444p12le", 12, mx
    return "yuv444p16le", 16, mx


def make_reference(src_cmd, pixfmt, out_mkv, w=W_SRC, h=H_SRC, fps=FPS_SRC):
    """src_cmd: 원본 raw yuv를 stdout으로 내보내는 명령 (curl 등)"""
    vf = (f"select='not(mod(n\\,2))',setpts=N/({FPS_OUT}*TB),"
          f"scale={W_OUT}:{H_OUT}:flags=area,format=yuv420p")
    sh(f"{src_cmd} | ffmpeg -y -loglevel error -f rawvideo -pix_fmt {pixfmt} -s {w}x{h} -framerate {fps} -i pipe:0 "
       f"-vf \"{vf}\" -r {FPS_OUT} -c:v ffv1 -level 3 -pix_fmt yuv420p \"{out_mkv}\"")


def encode_h264(ref_mkv, kbps, out_mp4, tmpdir):
    common = (f"-c:v libx264 -preset medium -profile:v high -pix_fmt yuv420p -b:v {kbps}k "
              f"-g {GOP} -keyint_min {GOP} -sc_threshold 0 -an")
    plog = os.path.join(tmpdir, f"x264_{kbps}")
    sh(f"ffmpeg -y -loglevel error -i \"{ref_mkv}\" {common} -pass 1 -passlogfile \"{plog}\" -f null /dev/null")
    sh(f"ffmpeg -y -loglevel error -i \"{ref_mkv}\" {common} -pass 2 -passlogfile \"{plog}\" \"{out_mp4}\"")


def probe(path):
    js = json.loads(sh(f"ffprobe -v error -select_streams v:0 -count_frames "
                       f"-show_entries stream=codec_name,width,height,nb_read_frames,avg_frame_rate,pix_fmt "
                       f"-show_entries format=duration,size,bit_rate -of json \"{path}\""))
    s, f = js["streams"][0], js["format"]
    n, d = map(float, s["avg_frame_rate"].split("/"))
    fps = n / d
    frames = int(s.get("nb_read_frames", 0))
    dur = frames / fps if frames else float(f["duration"])
    kbps = float(f["size"]) * 8 / dur / 1000  # 파일 크기 기준 실측
    return dict(codec=s["codec_name"], width=s["width"], height=s["height"], fps=round(fps, 3),
                frames=frames, kbps=round(kbps, 1), bpp=round(kbps * 1000 / (s["width"] * s["height"] * fps), 5),
                pix_fmt=s.get("pix_fmt"))


def keyframe_count(path):
    out = sh(f"ffprobe -v error -select_streams v:0 -show_entries frame=pict_type -of csv=p=0 \"{path}\"")
    t = [x.strip().strip(",") for x in out.split() if x.strip().strip(",")]
    return dict(I=t.count("I"), P=t.count("P"), B=t.count("B"))


# ---------------------------------------------------------------- 라벨 변환
_FRAME_KEYS = ("frame_id", "frame_index", "frame_idx", "frame", "frame_number", "frame_no")


def _frame_of(img, order_idx):
    for k in _FRAME_KEYS:
        if k in img and isinstance(img[k], (int, float)):
            return int(img[k]), k
    fn = str(img.get("file_name", ""))
    nums = re.findall(r"\d+", os.path.basename(fn))
    if nums:
        return int(nums[-1]), "file_name"
    return order_idx, "order"


def convert_ann(raw, n_src_frames):
    """COCO형 라벨을 짝수 프레임(30fps)·720p 좌표로 변환. 형식이 다르면 None과 사유 반환"""
    if not (isinstance(raw, dict) and "images" in raw and "annotations" in raw):
        return None, "COCO 형식 아님 (images/annotations 키 없음)"
    imgs = sorted(raw["images"], key=lambda x: x["id"])
    fr, src = {}, collections.Counter()
    for i, im in enumerate(imgs):
        f, how = _frame_of(im, i); fr[im["id"]] = f; src[how] += 1
    frames = sorted(set(fr.values()))
    base = frames[0] if frames else 0  # 0 또는 1부터 시작하는 경우 모두 처리
    if base not in (0, 1):
        return None, f"프레임 번호 시작값 이상: {base}"
    note = f"frame source={dict(src)}, base={base}, frames {len(frames)} (원본 {n_src_frames})"
    new_imgs, idmap = [], {}
    for im in imgs:
        f = fr[im["id"]] - base
        if f % 2:  # 홀수 프레임 버림 (60->30fps)
            continue
        nid = f // 2
        idmap[im["id"]] = nid
        new_imgs.append({"id": nid, "file_name": f"{nid:06d}.png", "width": W_OUT, "height": H_OUT,
                         "src_frame": f})
    anns, k = [], 0
    for a in raw["annotations"]:
        if a.get("image_id") not in idmap or "bbox" not in a:
            continue
        x, y, w, h = [v / SCALE for v in a["bbox"]]
        if w <= 0 or h <= 0:
            continue
        k += 1
        anns.append({"id": k, "image_id": idmap[a["image_id"]], "category_id": a["category_id"],
                     "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                     "area": round(w * h, 2), "iscrowd": a.get("iscrowd", 0),
                     **({"track_id": a["track_id"]} if "track_id" in a else {})})
    out = {"images": sorted(new_imgs, key=lambda x: x["id"]), "annotations": anns,
           "categories": raw.get("categories", [])}
    return out, note


def ann_stats(coco):
    s = m = l = 0
    for a in coco["annotations"]:
        ar = a["area"]
        s += ar < 32 ** 2; m += 32 ** 2 <= ar < 96 ** 2; l += ar >= 96 ** 2
    cats = {c["id"]: c["name"] for c in coco["categories"]}
    cnt = collections.Counter(cats.get(a["category_id"], a["category_id"]) for a in coco["annotations"])
    n = max(1, len(coco["annotations"]))
    return dict(label_images=len(coco["images"]), label_imgs_with_box=len({a["image_id"] for a in coco["annotations"]}),
                boxes=len(coco["annotations"]), small=s, medium=m, large=l, small_ratio=round(s / n, 3),
                top_classes=json.dumps(dict(cnt.most_common(6)), ensure_ascii=False))


def preview(ref_mkv, out_dir, n_frames):
    sh(f"ffmpeg -y -loglevel error -i \"{ref_mkv}\" -vf \"select='eq(n\\,0)'\" -frames:v 1 \"{out_dir}/first.png\"")
    last = max(0, n_frames - 1)
    sh(f"ffmpeg -y -loglevel error -i \"{ref_mkv}\" -vf \"select='eq(n\\,{last})'\" -frames:v 1 \"{out_dir}/last.png\"")
    from PIL import Image
    a = np.asarray(Image.open(f"{out_dir}/first.png").convert("L").resize((320, 180)), np.float32)
    b = np.asarray(Image.open(f"{out_dir}/last.png").convert("L").resize((320, 180)), np.float32)
    return round(float(np.abs(a - b).mean()), 2)


In [ ]:
import os, sys, json, time, csv, shutil, subprocess
import numpy as np, requests
sys.path.insert(0, "/kaggle/working"); import build_lib as B

# ffmpeg에 libx264·ffv1 있는지 확인
enc = subprocess.run("ffmpeg -hide_banner -encoders", shell=True, capture_output=True, text=True).stdout
assert "libx264" in enc and "ffv1" in enc, "ffmpeg에 libx264/ffv1 없음 -> 로그 전달"
print(subprocess.run("ffmpeg -version | head -1", shell=True, capture_output=True, text=True).stdout)

BASE = "https://tie-ultravideo.rd.tuni.fi/UVG-VCM"
OUT = "/kaggle/working/uvg_cctv"; os.makedirs(OUT, exist_ok=True)
RESUME_FROM = ""   # 이어서 할 때: 이전 버전 output의 uvg_cctv 경로 (예: /kaggle/input/.../uvg_cctv)

# 우선순위 순서 (CCTV 시점일 가능성이 높은 것부터). 시간 부족하면 뒤쪽은 다음 실행에서
SEQS = [("HighwayView", 600), ("CampusView", 600), ("JobFair", 600), ("OfficeWalk", 600),
        ("TrafficLights", 600), ("SpotRobot", 600), ("BoatReverse", 300), ("VolleyballGame", 600),
        ("FloorballTrain", 600), ("FloorballGame", 420), ("WinterDrive", 600), ("HighwayDrive", 600)]
TIME_BUDGET_H = 10.5   # Kaggle 12시간 제한 전에 안전하게 멈춤
FSIZE = B.W_SRC * B.H_SRC * 3 * 2

In [ ]:
if RESUME_FROM and os.path.isdir(RESUME_FROM):
    for d in os.listdir(RESUME_FROM):
        src = os.path.join(RESUME_FROM, d)
        if os.path.isdir(src) and os.path.exists(os.path.join(src, "done.json")) and not os.path.exists(os.path.join(OUT, d)):
            shutil.copytree(src, os.path.join(OUT, d)); print("resume copy", d)

t_start = time.time()
for name, n in SEQS:
    d = os.path.join(OUT, name); os.makedirs(d, exist_ok=True)
    if os.path.exists(os.path.join(d, "done.json")):
        print(name, "이미 완료 - 건너뜀"); continue
    if (time.time() - t_start) / 3600 > TIME_BUDGET_H:
        print("시간 예산 초과 - 여기서 중단:", name); break
    t0 = time.time()
    try:
        url = f"{BASE}/{name}/{name}_{B.W_SRC}x{B.H_SRC}_60fps_yuv444_16bits_{n}.yuv"
        # 1) 첫 프레임으로 실제 비트 깊이 확인
        r = requests.get(url, headers={"Range": f"bytes=0-{FSIZE-1}"}, timeout=600); r.raise_for_status()
        assert len(r.content) == FSIZE, f"Range 응답 크기 이상 {len(r.content)}"
        pixfmt, bits, mx = B.detect_pixfmt(np.frombuffer(r.content, dtype="<u2"))
        print(f"\n=== {name}: max code {mx} -> {pixfmt}")
        # 2) 원본 스트리밍 -> 720p30 무손실 기준 영상
        ref = os.path.join(d, "ref_720p30.mkv")
        B.make_reference(f'curl -sSfL --retry 5 "{url}"', pixfmt, ref)
        pr = B.probe(ref)
        assert pr["frames"] == n // 2, f"기준 영상 프레임 수 {pr['frames']} != {n//2} (다운로드 끊김?)"
        static = B.preview(ref, d, pr["frames"])
        meta = dict(seq=name, src_frames=n, src_pixfmt=pixfmt, src_max_code=mx, ref_frames=pr["frames"],
                    ref_mbps=round(pr["kbps"] / 1000, 1), first_last_diff=static)
        # 3) H.264 단계별 압축
        for k in B.BITRATES_K:
            mp4 = os.path.join(d, f"h264_{k}k.mp4")
            B.encode_h264(ref, k, mp4, "/tmp")
            p = B.probe(mp4); kf = B.keyframe_count(mp4)
            assert p["frames"] == pr["frames"]
            meta[f"{k}k_kbps"] = p["kbps"]; meta[f"{k}k_bpp"] = p["bpp"]; meta[f"{k}k_IPB"] = f"{kf['I']}/{kf['P']}/{kf['B']}"
            print(f"  {k}k -> 실측 {p['kbps']} kbps, bpp {p['bpp']}, I/P/B {kf}")
        # 4) 라벨
        rr = requests.get(f"{BASE}/{name}/{name}_annotations.json", timeout=600); rr.raise_for_status()
        raw = rr.json(); json.dump(raw, open(os.path.join(d, "ann_raw.json"), "w"))
        coco, note = B.convert_ann(raw, n)
        meta["ann_note"] = note
        if coco:
            json.dump(coco, open(os.path.join(d, "ann_720p30_coco.json"), "w"))
            meta.update(B.ann_stats(coco))
        else:
            print("  라벨 변환 실패:", note, "| top keys:", list(raw)[:10] if isinstance(raw, dict) else type(raw))
        print("  ", note)
        meta["minutes"] = round((time.time() - t0) / 60, 1)
        json.dump(meta, open(os.path.join(d, "done.json"), "w"), ensure_ascii=False, indent=1)
        print(f"  완료 {meta['minutes']}분 | 첫/끝 차이 {static} | 작은 객체 비율 {meta.get('small_ratio')}")
    except Exception as e:
        print(name, "ERROR:", str(e)[-1500:])

In [ ]:
rows = [json.load(open(os.path.join(OUT, d, "done.json"))) for d in sorted(os.listdir(OUT))
        if os.path.exists(os.path.join(OUT, d, "done.json"))]
keys = []
for r in rows:
    for k in r:
        if k not in keys: keys.append(k)
with open(os.path.join(OUT, "meta.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(rows)
# 미리보기 모음
from PIL import Image, ImageDraw
if rows:
    sheet = Image.new("RGB", (960, 290 * len(rows)), "white"); dr = ImageDraw.Draw(sheet)
    for i, r in enumerate(rows):
        for j, fn in enumerate(["first.png", "last.png"]):
            sheet.paste(Image.open(os.path.join(OUT, r["seq"], fn)).resize((480, 270)), (480 * j, 290 * i + 20))
        dr.text((5, 290 * i + 3), f"{r['seq']}  diff={r['first_last_diff']}  small={r.get('small_ratio')}", fill="black")
    sheet.save(os.path.join(OUT, "contact_sheet.png"))
print(len(rows), "개 시퀀스 완료")
os.system(f"du -sh {OUT}")
for r in rows: print(r)